## Import Libraries

In [29]:
import pandas as pd
import numpy as np

print("Libraries imported ✅")

Libraries imported ✅


## Load Cleaned data

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd().parent

file_path = (
    BASE_DIR
    / "datasets"
    / "processed_datasets"
    / "clustered_data.csv"
)

df = pd.read_csv(file_path)

df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,...,Month,Is_Weekend,Season,Crime_Severity,Primary_Type_Encoded,Location_Encoded,Lat_scaled,Long_scaled,Cluster_KMeans,Cluster_DBSCAN
0,3180479,2004-02-14 04:00:00,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,RESIDENCE,False,False,331,3,5,...,2,1,Winter,5.0,5,126,-0.806220,1.643164,1,-1
1,12549744,2021-11-24 15:05:00,THEFT,RETAIL THEFT,SMALL RETAIL STORE,False,False,2032,20,47,...,11,0,Fall,2.0,31,146,1.459480,0.049758,4,-1
2,3189525,2004-02-21 18:00:00,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,False,False,1611,16,41,...,2,1,Winter,2.0,6,126,1.745940,-2.145479,0,-1
3,3371426,2004-06-08 19:00:00,THEFT,$500 AND UNDER,STREET,False,False,1211,12,2,...,6,0,Summer,2.0,31,150,0.438992,0.070117,2,-1
4,7987090,2011-03-25 09:00:00,ASSAULT,SIMPLE,RESIDENCE,False,False,2132,2,4,...,3,0,Spring,4.0,1,126,-0.470595,1.432920,1,-1


In [ ]:
# file_path = "../datasets/processed_datasets/cleaned_data.csv"

# df = pd.read_csv(file_path)

# print("Data loaded ✅")

Data loaded ✅


## Basics Check

In [31]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,X Coordinate,Y Coordinate,Year,Latitude,Longitude
0,2588748,2003-02-14 06:17:00,BATTERY,SIMPLE,ALLEY,False,False,1324,12,27,24,1166236.0,1904446.0,2003,41.893379,-87.664923
1,4376996,2005-10-12 17:51:46,BATTERY,SIMPLE,SIDEWALK,False,False,722,7,6,69,1175998.0,1859224.0,2005,41.769072,-87.630429
2,11765740,2019-07-20 11:15:00,OTHER OFFENSE,ANIMAL ABUSE/NEGLECT,RESIDENCE,False,False,512,5,9,49,1179045.0,1835359.0,2019,41.703514,-87.619986
3,13858494,2025-06-05 08:00:00,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,RESIDENCE,False,False,623,6,6,69,1178022.0,1855082.0,2025,41.757660,-87.623136
4,8688341,2012-07-02 08:30:00,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,1324,12,26,24,1165948.0,1903575.0,2012,41.890995,-87.666005


In [32]:
df.shape

(494196, 16)

In [33]:
df.columns

Index(['ID', 'Date', 'Primary Type', 'Description', 'Location Description',
       'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area',
       'X Coordinate', 'Y Coordinate', 'Year', 'Latitude', 'Longitude'],
      dtype='object')

In [34]:
df.dtypes

ID                        int64
Date                     object
Primary Type             object
Description              object
Location Description     object
Arrest                     bool
Domestic                   bool
Beat                      int64
District                  int64
Ward                      int64
Community Area            int64
X Coordinate            float64
Y Coordinate            float64
Year                      int64
Latitude                float64
Longitude               float64
dtype: object

## Convert Data

In [35]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df.dtypes

ID                               int64
Date                    datetime64[ns]
Primary Type                    object
Description                     object
Location Description            object
Arrest                            bool
Domestic                          bool
Beat                             int64
District                         int64
Ward                             int64
Community Area                   int64
X Coordinate                   float64
Y Coordinate                   float64
Year                             int64
Latitude                       float64
Longitude                      float64
dtype: object

## Extract Time Features

In [36]:
df["Hour"] = df["Date"].dt.hour
df["Day_of_Week"] = df["Date"].dt.day_name()
df["Month"] = df["Date"].dt.month

In [37]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,X Coordinate,Y Coordinate,Year,Latitude,Longitude,Hour,Day_of_Week,Month
0,2588748,2003-02-14 06:17:00,BATTERY,SIMPLE,ALLEY,False,False,1324,12,27,24,1166236.0,1904446.0,2003,41.893379,-87.664923,6,Friday,2
1,4376996,2005-10-12 17:51:46,BATTERY,SIMPLE,SIDEWALK,False,False,722,7,6,69,1175998.0,1859224.0,2005,41.769072,-87.630429,17,Wednesday,10
2,11765740,2019-07-20 11:15:00,OTHER OFFENSE,ANIMAL ABUSE/NEGLECT,RESIDENCE,False,False,512,5,9,49,1179045.0,1835359.0,2019,41.703514,-87.619986,11,Saturday,7
3,13858494,2025-06-05 08:00:00,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,RESIDENCE,False,False,623,6,6,69,1178022.0,1855082.0,2025,41.757660,-87.623136,8,Thursday,6
4,8688341,2012-07-02 08:30:00,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,1324,12,26,24,1165948.0,1903575.0,2012,41.890995,-87.666005,8,Monday,7


## Create Weekend

In [38]:
df["Is_Weekend"] = df["Day_of_Week"].isin(["Saturday", "Sunday"]).astype(int)

df[["Day_of_Week", "Is_Weekend"]].head()

,Day_of_Week,Is_Weekend
0,Friday,0
1,Wednesday,0
2,Saturday,1
3,Thursday,0
4,Monday,0


## Create Season Feature

In [39]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

df["Season"] = df["Month"].apply(get_season)

df[["Month", "Season"]].head()

,Month,Season
0,2,Winter
1,10,Fall
2,7,Summer
3,6,Summer
4,7,Summer


## Crime severity score


In [40]:
severity_map = {
    "HOMICIDE": 5,
    "CRIM SEXUAL ASSAULT": 5,
    "ROBBERY": 4,
    "ASSAULT": 4,
    "BATTERY": 4,
    "BURGLARY": 3,
    "MOTOR VEHICLE THEFT": 3,
    "THEFT": 2,
    "CRIMINAL DAMAGE": 2,
    "NARCOTICS": 1
}

In [41]:
df["Crime_Severity"] = df["Primary Type"].map(severity_map)
df["Crime_Severity"] = df["Crime_Severity"].fillna(1)

df[["Primary Type", "Crime_Severity"]].head()

,Primary Type,Crime_Severity
0,BATTERY,4.0
1,BATTERY,4.0
2,OTHER OFFENSE,1.0
3,DECEPTIVE PRACTICE,1.0
4,BURGLARY,3.0


## Encode Categorical Columns

#### Label encoding

In [42]:
from sklearn.preprocessing import LabelEncoder

le_primary = LabelEncoder()
df["Primary_Type_Encoded"] = le_primary.fit_transform(df["Primary Type"])

In [43]:
le_location = LabelEncoder()
df["Location_Encoded"] = le_location.fit_transform(df["Location Description"])

In [44]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,...,Latitude,Longitude,Hour,Day_of_Week,Month,Is_Weekend,Season,Crime_Severity,Primary_Type_Encoded,Location_Encoded
0,2588748,2003-02-14 06:17:00,BATTERY,SIMPLE,ALLEY,False,False,1324,12,27,...,41.893379,-87.664923,6,Friday,2,0,Winter,4.0,2,15
1,4376996,2005-10-12 17:51:46,BATTERY,SIMPLE,SIDEWALK,False,False,722,7,6,...,41.769072,-87.630429,17,Wednesday,10,0,Fall,4.0,2,145
2,11765740,2019-07-20 11:15:00,OTHER OFFENSE,ANIMAL ABUSE/NEGLECT,RESIDENCE,False,False,512,5,9,...,41.703514,-87.619986,11,Saturday,7,1,Summer,1.0,23,126
3,13858494,2025-06-05 08:00:00,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,RESIDENCE,False,False,623,6,6,...,41.757660,-87.623136,8,Thursday,6,0,Summer,1.0,9,126
4,8688341,2012-07-02 08:30:00,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,1324,12,26,...,41.890995,-87.666005,8,Monday,7,0,Summer,3.0,3,17


## Geographic Features

In [45]:
df = df[
    (df["Latitude"] > 41.6) & (df["Latitude"] < 42.1) &
    (df["Longitude"] > -88.0) & (df["Longitude"] < -87.5)
]

In [46]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df[["Lat_scaled", "Long_scaled"]] = scaler.fit_transform(df[["Latitude", "Longitude"]])

In [47]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,...,Hour,Day_of_Week,Month,Is_Weekend,Season,Crime_Severity,Primary_Type_Encoded,Location_Encoded,Lat_scaled,Long_scaled
0,2588748,2003-02-14 06:17:00,BATTERY,SIMPLE,ALLEY,False,False,1324,12,27,...,6,Friday,2,0,Winter,4.0,2,15,0.586018,0.105230
1,4376996,2005-10-12 17:51:46,BATTERY,SIMPLE,SIDEWALK,False,False,722,7,6,...,17,Wednesday,10,0,Fall,4.0,2,145,-0.851908,0.688908
2,11765740,2019-07-20 11:15:00,OTHER OFFENSE,ANIMAL ABUSE/NEGLECT,RESIDENCE,False,False,512,5,9,...,11,Saturday,7,1,Summer,1.0,23,126,-1.610244,0.865632
3,13858494,2025-06-05 08:00:00,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,RESIDENCE,False,False,623,6,6,...,8,Thursday,6,0,Summer,1.0,9,126,-0.983914,0.812327
4,8688341,2012-07-02 08:30:00,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,1324,12,26,...,8,Monday,7,0,Summer,3.0,3,17,0.558442,0.086910


In [48]:
df.columns

Index(['ID', 'Date', 'Primary Type', 'Description', 'Location Description',
       'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area',
       'X Coordinate', 'Y Coordinate', 'Year', 'Latitude', 'Longitude', 'Hour',
       'Day_of_Week', 'Month', 'Is_Weekend', 'Season', 'Crime_Severity',
       'Primary_Type_Encoded', 'Location_Encoded', 'Lat_scaled',
       'Long_scaled'],
      dtype='object')

## Select Features for ML

In [49]:
features = [
    "Lat_scaled",
    "Long_scaled",
    "Hour",
    "Month",
    "Is_Weekend",
    "Crime_Severity",
    "Primary_Type_Encoded",
    "Location_Encoded"
]

X = df[features]

X.head()

,Lat_scaled,Long_scaled,Hour,Month,Is_Weekend,Crime_Severity,Primary_Type_Encoded,Location_Encoded
0,0.586018,0.105230,6,2,0,4.0,2,15
1,-0.851908,0.688908,17,10,0,4.0,2,145
2,-1.610244,0.865632,11,7,1,1.0,23,126
3,-0.983914,0.812327,8,6,0,1.0,9,126
4,0.558442,0.086910,8,7,0,3.0,3,17


## Final Check

In [50]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,...,Hour,Day_of_Week,Month,Is_Weekend,Season,Crime_Severity,Primary_Type_Encoded,Location_Encoded,Lat_scaled,Long_scaled
0,2588748,2003-02-14 06:17:00,BATTERY,SIMPLE,ALLEY,False,False,1324,12,27,...,6,Friday,2,0,Winter,4.0,2,15,0.586018,0.105230
1,4376996,2005-10-12 17:51:46,BATTERY,SIMPLE,SIDEWALK,False,False,722,7,6,...,17,Wednesday,10,0,Fall,4.0,2,145,-0.851908,0.688908
2,11765740,2019-07-20 11:15:00,OTHER OFFENSE,ANIMAL ABUSE/NEGLECT,RESIDENCE,False,False,512,5,9,...,11,Saturday,7,1,Summer,1.0,23,126,-1.610244,0.865632
3,13858494,2025-06-05 08:00:00,DECEPTIVE PRACTICE,CREDIT CARD FRAUD,RESIDENCE,False,False,623,6,6,...,8,Thursday,6,0,Summer,1.0,9,126,-0.983914,0.812327
4,8688341,2012-07-02 08:30:00,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,1324,12,26,...,8,Monday,7,0,Summer,3.0,3,17,0.558442,0.086910


In [51]:
X.isnull().sum()

Lat_scaled              0
Long_scaled             0
Hour                    0
Month                   0
Is_Weekend              0
Crime_Severity          0
Primary_Type_Encoded    0
Location_Encoded        0
dtype: int64

In [52]:
X.shape

(494184, 8)

## Save Featured Data

In [53]:
df.to_csv("../datasets/processed_datasets/feature_data.csv", index=False)
print("Feature engineered data saved ✅")

Feature engineered data saved ✅
